## Conformal Risk Control for Startup Success Prediction through Machine Learning

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv 
import yaml

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer

from src.encoding import apply_frequency_encoding, fit_frequency_encoding
from src.models import build
from src.utils import (set_seed,
                       run_conformal_fnr_experiment,
                       run_conformal_fnr_group_calibrated, plot_conformal_fnr,
                       assign_model_colors, fnr_by_group_table,
                       crc_summary_tables, plot_crc_metric_boxplots,
                       plot_fnr_by_group_boxplots, plot_fnr_by_group_lines)


from sklearn.metrics import accuracy_score, roc_auc_score
import wandb
import matplotlib.pyplot as plt 


import scipy.integrate
if not hasattr(np, 'trapz'):
    np.trapz = scipy.integrate.trapezoid

load_dotenv()

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)


timeWindow=int(config['time_window'])

lastYear=int(config['last_year'])

conformal_store = {}
conformal_pool_store = {}

## Split, Imputation and Scaling

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset"])

X = dataset.drop(['CompanyID', 'Target'], axis=1)
y = dataset['Target']

train_frac = config['train_frac']
cal_frac   = config['calibration_frac']
val_frac   = config['validation_frac']
test_frac  = config['test_frac']
seed       = config['random_seed']

# Partial split (stratified): train / validation /  calibration & test = 50% / 10% / 20% + 20%.
# Final calibration-test split is done later in the multi-seed robustness check, which re-splits the calibration-test pool 50/50.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, train_size=train_frac, stratify=y, random_state=seed)

X_val, X_cal_test, y_val, y_cal_test = train_test_split( X_tmp, y_tmp, train_size=val_frac, stratify=y_tmp, random_state=seed)


# Frequency encoding of the categorical columns, fitted on train only: a category
# share computed before the split would partly come from the rows it encodes.
# `dataset` keeps the raw names, used for the sector subgroups in RQ2/RQ3.
enc_cfg = config["frequency_encoding"]
X_train, X_val, X_cal_test = X_train.copy(), X_val.copy(), X_cal_test.copy()
for column in enc_cfg["columns"]:
    encoding = fit_frequency_encoding(X_train[column],
                                      min_frequency=enc_cfg["min_frequency"],
                                      other_label=enc_cfg["other_label"])
    for frame in (X_train, X_val, X_cal_test):
        frame[column] = apply_frequency_encoding(frame[column], encoding,
                                                 other_label=enc_cfg["other_label"])

# Imputation (fit on train, apply to all splits)
imputer = KNNImputer(n_neighbors=5)
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_cal_test_imp   = imputer.transform(X_cal_test)
    
# Scaling (fit on imputed train, apply to all splits)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_cal_test_scaled   = scaler.transform(X_cal_test_imp)

# Matrices in the layout the model wrappers read (Model.matrices in
# src/models/base.py): the "test" slot is the calibration+test pool.
split = {
    "X_train_imp": X_train_imp, "X_val_imp": X_val_imp, "X_test_imp": X_cal_test_imp,
    "X_train_scaled": X_train_scaled, "X_val_scaled": X_val_scaled,
    "X_test_scaled": X_cal_test_scaled,
}



print(f"Total: {len(y)} | Train: {len(y_train)}| Val: {len(y_val)} | Cal+Test: {len(y_cal_test)}")

## Sweep creation

In [ ]:

entity=os.getenv("entity")
project=os.getenv("project")

# Initialize a new sweep
sweep_id = wandb.sweep(config["sweep_settings"],entity=entity, project=project)


Definition of the training function to be called by wandb agent.

It will be called multiple times by the wandb agent, each time with a different set of hyperparameters defined in the sweep configuration.

The function trains the specified model, evaluates it on the validation and test sets, logs metrics and plots to Weights & Biases, and computes SHAP values for interpretability.

In [ ]:

def train():
    with wandb.init():
        wandb_config = wandb.config

        # Reproducibility: seed Python, NumPy, PyTorch (CPU+CUDA), cuDNN.
        # The wrappers also pass the seed as random_state to every estimator.
        set_seed(config['random_seed'])
        
        # Each family picks its matrices (imputed, or imputed and scaled) and
        # its own fit; see src/models/.
        model = build(wandb_config.model_type, wandb_config, seed=config['random_seed'])
        X_tr, X_v, X_ct = model.matrices(split)
        model.fit(X_tr, y_train, X_v, y_val)

        probs_train, preds_train = model.score(X_tr)
        labels_train = y_train

        probs_cal_test = model.predict_proba(X_ct)
        labels_cal_test = y_cal_test

        # Training-set fit indicators (constant across seeds): overfitting check.
        acc_train = accuracy_score(labels_train, preds_train)
        try:
            auc_train = roc_auc_score(labels_train, probs_train)
        except ValueError:
            auc_train = float("nan")

        wandb.log({"accuracy_train": acc_train, "AUC_train": auc_train})
        print(f"Train accuracy: {acc_train:.3f} | Train AUC: {auc_train:.3f}")

        # ── CONFORMAL RISK CONTROL ON FNR ──
        # Per-run metrics come from the conformal experiment on the cal+test
        # pool: for every seed a base row (no calibration, 0.5 threshold) and
        # one row per target alpha (conformal threshold t*). Metrics: AUC
        # (base only, threshold-free), accuracy, precision, recall, f1, FNR, FPR.
        conf_cfg = config.get("conformal", {})
        if conf_cfg.get("enabled", False):

            df_conf = run_conformal_fnr_experiment(
                probs_cal_test, labels_cal_test,
                alphas=conf_cfg["alpha"],
                seeds=conf_cfg["seeds"],
            )

            fig_conf = plot_conformal_fnr(df_conf, model_name=wandb_config.model_type)

            # Run-level scalars (mean over seeds): pre- (base, 0.5) and
            # post-calibration (t* per alpha).
            base = df_conf[~df_conf["calibrated"]]
            conf_log = {
                "conformal/AUC":            float(base["AUC"].mean()),
                "conformal/base/accuracy":  float(base["accuracy"].mean()),
                "conformal/base/precision": float(base["precision"].mean()),
                "conformal/base/recall":    float(base["recall"].mean()),
                "conformal/base/f1":        float(base["f1"].mean()),
                "conformal/base/FNR":       float(base["FNR"].mean()),
                "conformal/base/FPR":       float(base["FPR"].mean()),
            }
            for a in conf_cfg["alpha"]:
                sub = df_conf[df_conf["calibrated"] & (df_conf["alpha"] == a)]
                conf_log[f"conformal/conf/a{a}/accuracy"]  = float(sub["accuracy"].mean())
                conf_log[f"conformal/conf/a{a}/precision"] = float(sub["precision"].mean())
                conf_log[f"conformal/conf/a{a}/recall"]    = float(sub["recall"].mean())
                conf_log[f"conformal/conf/a{a}/f1"]        = float(sub["f1"].mean())
                conf_log[f"conformal/conf/a{a}/FNR"]       = float(sub["FNR"].mean())
                conf_log[f"conformal/conf/a{a}/FPR"]       = float(sub["FPR"].mean())
                conf_log[f"conformal/conf/a{a}/threshold"] = float(sub["threshold"].mean())

            wandb.log({
                "conformal_fnr_table": wandb.Table(dataframe=df_conf),
                "conformal_fnr_plot": wandb.Image(fig_conf),
                **conf_log,
            })
            plt.close(fig_conf)

            mean_fnr = df_conf[df_conf["calibrated"]].groupby("alpha")["FNR"].mean()
            print("Conformal FNR control (mean realized FNR per alpha):")
            print(mean_fnr.to_string())

            conformal_store[wandb_config.model_type] = df_conf
            conformal_pool_store[wandb_config.model_type] = probs_cal_test

number_of_runs = len(config["sweep_settings"]["parameters"]["model_type"]["values"])

wandb.agent(sweep_id, function=train, count=number_of_runs)

## RQ1 — Global-level CRC

For each model: the aggregated (mean over seeds) realized test FNR per target
`alpha` and the corresponding plot. Points on/under the `y = x` diagonal mean the
risk `E[FNR_test] <= alpha` is controlled across calibration/test re-splits.

In [ ]:
for model_type, df_conf in conformal_store.items():
    print(f"=== {model_type.upper()} ===")
    base, conf = crc_summary_tables(df_conf)
    print("Pre-calibration metrics (mean over seeds):")
    display(base)
    print("Post-calibration metrics per target alpha (mean over seeds):")
    display(conf)
    plot_conformal_fnr(df_conf, model_name=model_type)
    plt.show()

**What the plot shows.** Each panel reports the realized test False Negative
Rate (FNR, $y$-axis) as a function of the target risk level $\alpha$ ($x$-axis)
for one model. The scattered points are the per-seed outcomes over the
calibration/test re-splits; the solid line is the mean realized FNR across seeds
and the shaded band its $\pm 1$ standard deviation. The dashed line is the
$y = x$ reference: points lying on or below it indicate that the realized risk
does not exceed the target, i.e. the FNR is controlled at level $\alpha$.

**Observations.** For all three models the mean realized FNR lies on or below
the $y = x$ diagonal (realized FNR $\approx$ or $<\alpha$). Although individual
re-splits occasionally exceed the requested $\alpha$, the target risk is
controlled *marginally*: the guarantee $\mathbb{E}[\mathrm{FNR}_\mathrm{test}]
\le \alpha$ holds in expectation over the calibration/test splits rather than for
every single split. The seed-to-seed dispersion (band width) grows with
$\alpha$.

**Interpretation.** This is the expected behaviour of Conformal Risk Control
(Angelopoulos et al., 2024): the guarantee is distribution-free but marginal and
holds in expectation, so it characterises the pool as a whole rather than any
individual split or subgroup (the latter is examined in RQ2). Because the
control is a property of the calibration procedure rather than of the
classifier, the three models behave almost identically here; their base AUC
(~0.79–0.80) remains the only meaningful quality discriminator.


### Global-level CRC boxplots

Per-metric seed-to-seed distribution. One figure per metric; boxes grouped by
model, the first (gray) box the uncalibrated baseline (0.5) then one box per
target `alpha` at the conformal threshold.

In [ ]:
# Global-level CRC: per-metric boxplots over seeds (one representative frame per
# model; the pool is fixed so every model's frame carries the same seeds).
model_df = {}
for model_type, df_conf in conformal_store.items():
    model_df.setdefault(model_type, df_conf)
plot_crc_metric_boxplots(model_df)

**What the plots show.** One figure per metric. Models are placed side by side
(RF, LGB, MLP); within each model the first (grey) box is the uncalibrated
baseline (0.5 threshold), followed by one box per target $\alpha$ (blue ramp,
from light $\alpha=0.05$ to dark $\alpha=0.25$). Each box is the seed-to-seed
distribution of the metric.

**Accuracy.** Increases monotonically, from ~0.50–0.52 ($\alpha=0.05$) to
~0.69–0.71 ($\alpha=0.25$), remaining just below the baseline (~0.72–0.73). Its
trend essentially follows the FPR (once false negatives are few, accuracy is
driven mainly by false positives): lowering the threshold to guard against FNs
sacrifices accuracy, which is almost fully recovered only at the most permissive
target.

**Precision.** Increases from ~0.38–0.40 ($\alpha=0.05$) to ~0.52–0.53
($\alpha=0.25$), remaining **always below** the baseline (~0.55–0.57): low
thresholds $\Rightarrow$ many predicted positives $\Rightarrow$ many FPs
$\Rightarrow$ low precision; as $\alpha$ increases the threshold rises and
precision recovers, without ever reaching the baseline.

**Recall.** Mirror image of the FNR (recall $= 1 - \mathrm{FNR}$): **decreases**
from ~0.95 ($\alpha=0.05$) to ~0.755 ($\alpha=0.25$), remaining **always above**
the baseline (~0.69–0.72). Buying FNR control means buying recall.

**F1.** The **flattest** metric: it rises slightly from ~0.55 to ~0.61–0.63 and
at $\alpha=0.25$ **essentially matches the baseline** (~0.62). The recall gain
and the precision loss almost cancel out; the best operating point in F1 terms is
therefore high $\alpha$, where one attains the same F1 as the default **but with
a guaranteed, lower FNR**.

**FPR (the main cost).** The boxes **decrease** as $\alpha$ increases, from
~0.69–0.71 ($\alpha=0.05$) to ~0.31–0.34 ($\alpha=0.25$), but remain **always
above** the baseline (~0.25–0.28). At small $\alpha$ the false-positive cost is
large (nearly three times the baseline: ~0.70 vs ~0.25), returning close to the
baseline only at $\alpha=0.25$. This is where FNR control is "paid for".

**Summary of the patterns.** All metrics move **monotonically and consistently**
with $\alpha$ (recall/FPR decrease, precision/accuracy/F1 increase, FNR tracks
the target) and pivot around a single knob: $\alpha$ selects the operating point
on the trade-off curve. Relative to the **uncalibrated** model, CRC shifts the
operating regime toward high-recall/low-FNR at the cost of FP/precision/accuracy;
as $\alpha$ grows, all metrics approach the baseline again except the FNR, which
remains guaranteed and lower. In every panel the uncalibrated box thus reads as a
high-precision/high-accuracy model with an **uncontrolled FNR** (~0.30).


## RQ2 — Subgroups under global-level CRC

Characterization of subgroups when the FNR is controlled **marginally** (a single
global threshold). For each model the tables and plots report the realized FNR per
industry sector and per founding-year cohort (`YearFounded`); where `FNR_marg`
sits above the target `alpha`, that subgroup is under-covered by marginal-only
control.

In [ ]:
# Group labels on the conformal pool (calibration + test rows, in pool order).
# Reused across models, since the pool is fixed.
pool_index   = X_cal_test.index
cohort_pool  = dataset.loc[pool_index, "YearFounded"].to_numpy()
labels_pool  = y_cal_test.to_numpy()

# Industry sector by name, read off the raw column (the features carry its
# train-fitted frequency encoding instead). The few rows without a sector form an
# "Unknown" group, flagged infeasible by min_group.
sector_pool = dataset.loc[pool_index, "PrimaryIndustrySector"].fillna("Unknown").to_numpy()

alphas = config["conformal"]["alpha"]
seeds  = config["conformal"]["seeds"]

model_color = assign_model_colors(list(conformal_pool_store))

# Compute the per-(model, dimension) tables once; keep the raw per-seed marginal
# AND conditional frames (the latter feeds RQ3). Sector first, then cohort.
subgroup_dims = [("Industry sector", sector_pool),
                 ("YearFounded cohort", cohort_pool)]
subgroup_tables = {}
subgroup_raw_marg = {}
subgroup_raw_cond = {}
for model_type, probs_pool in conformal_pool_store.items():
    for dim_name, groups in subgroup_dims:
        tab, raw_marg, raw_cond = fnr_by_group_table(
            probs_pool, labels_pool, groups, alphas, seeds, min_group=20)
        subgroup_tables[(model_type, dim_name)] = tab
        subgroup_raw_marg[(model_type, dim_name)] = raw_marg
        subgroup_raw_cond[(model_type, dim_name)] = raw_cond

### Marginal coverage per subgroup

A positive `FNR_marg - alpha` is the per-group coverage gap left by the single
marginal threshold (blank when the group is at/under `alpha`).

In [ ]:
# Industry sector — table then grouped boxplot (models side by side, one figure
# per alpha).
marg_cols = ["group", "alpha", "FNR_marg", "FNR_marg - alpha",
             "n_pool", "n_pos_pool"]
dim_name = "Industry sector"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} ===")
    display(tab[marg_cols].rename(columns={"group": dim_name}))

sector_raw = {m: raw for (m, d), raw in subgroup_raw_marg.items() if d == dim_name}
plot_fnr_by_group_boxplots(sector_raw, dim_name, model_color,
                           ylabel="realized FNR (marginal)")

**What the plots show.** One figure per target `alpha`. The $x$-axis lists the
industry sectors; within each sector cluster there is one box per model (RF, LGB,
MLP), each reporting the seed-to-seed distribution of the realized FNR over the
calibration/test re-splits. The dashed red line marks the target `alpha`: boxes
lying **above** it are under-covered (realized FNR $>$ target), boxes **below**
it over-covered.

- **Under-covered** (FNR_marg $-\ \alpha > 0$, box above the line): chiefly

  - **B2C**: gap ~0.035 at `alpha=0.05`, growing to ~0.10 at `alpha=0.25`
  - **B2B** to a lesser extent (gap ~0.01→0.08).

- **Over-covered** (FNR_marg $\ll \alpha$):
  - **Financial Services** (~0.008),
  - Energy,
  - Healthcare,
  - Materials and Resources,
  - IT

The models behave similarly. The sector showing the largest variability is
**Energy**, especially at the lower `alpha` values, followed by Materials and
Resources. Energy and Materials are the **smallest** sectors in the pool (~295
and ~242 examples, ~100 and ~95 positives respectively); with few positives, each
calibration/test re-split shifts the realized FNR substantially, producing tall
boxes. The larger sectors, in contrast, have compact and stable boxes. This is
therefore a **sample-size** effect, not genuine model instability on
Energy/Materials.

At `alpha=0.05`, **Financial Services** attains a mean FNR $\approx 0.008$
(essentially null). The global conformal threshold at that `alpha` is very
permissive ($t^{*} \approx 0.21\text{–}0.25$) and in this sector almost all true
positives receive a score above the threshold: the group is "easy" and strongly
over-covered.


In [ ]:
# YearFounded cohort — table, grouped boxplot, then per-model line plot.
marg_cols = ["group", "alpha", "FNR_marg", "FNR_marg - alpha",
             "n_pool", "n_pos_pool"]
dim_name = "YearFounded cohort"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} ===")
    display(tab[marg_cols].rename(columns={"group": dim_name}))

cohort_raw = {m: raw for (m, d), raw in subgroup_raw_marg.items() if d == dim_name}
plot_fnr_by_group_boxplots(cohort_raw, dim_name, model_color,
                           ylabel="realized FNR (marginal)")
plot_fnr_by_group_lines(cohort_raw, dim_name, model_color,
                        ylabel="realized FNR (marginal)")

**What the plots show.** Two complementary views, one figure per target `alpha`.
The grouped boxplot places the founding-year cohorts on the $x$-axis, with one
box per model (RF, LGB, MLP) reporting the seed-to-seed distribution of the
realized FNR. The line plot shows, for each model, the mean realized FNR across
seeds (solid line) within a $\pm 1$ standard-deviation band, as a function of the
cohort year. In both views the dashed line marks the target `alpha`: cohorts
above it are under-covered, cohorts below it over-covered.

The per-model line plot reveals a clear **temporal gradient**:

- The **oldest** cohorts (2008–2009) are strongly over-covered: FNR $\approx 0$
  at almost all `alpha` values.

- The **most recent** cohort, **2017**, is **systematically under-covered**: the
  FNR_marg $-\ \alpha$ gap ranges from ~0.010 (`alpha=0.05`) up to ~0.06
  (`alpha=0.25`), the largest of all. Cohorts 2010 and 2012–2014 also remain
  slightly under-covered.


## RQ3 — Group-level CRC

Same outputs as RQ1 and RQ2 but after **group-level (Mondrian) calibration**: a
separate conformal threshold is calibrated per subgroup. Grouping is applied both
by industry sector and by founding-year cohort.

### As RQ1 — global metrics under group-level calibration

For each grouping, per-group thresholds are applied and the six metrics are
measured on the whole (pooled) test half, so the tables/boxplots are directly
comparable to RQ1.

In [ ]:
# Group-calibrated global CRC per (model, grouping). Reuses the same pool.
group_cal_store = {}
for model_type, probs_pool in conformal_pool_store.items():
    for dim_name, groups in subgroup_dims:
        group_cal_store[(model_type, dim_name)] = run_conformal_fnr_group_calibrated(
            probs_pool, labels_pool, groups, alphas, seeds, min_group=20)

In [ ]:
# Summary tables (pre/post) per model, one block per grouping.
for dim_name, _ in subgroup_dims:
    print(f"################  Group-level CRC calibrated by {dim_name}  ################")
    for (model_type, d), df_gc in group_cal_store.items():
        if d != dim_name:
            continue
        print(f"=== {model_type.upper()} — calibrated by {dim_name} ===")
        base, conf = crc_summary_tables(df_gc)
        print("Pre-calibration metrics (mean over seeds):")
        display(base)
        print("Post-calibration metrics per target alpha (mean over seeds):")
        display(conf)

In [ ]:
# Per-metric global boxplots, one set of figures per grouping.
for dim_name, _ in subgroup_dims:
    print(f"################  Group-level CRC calibrated by {dim_name}  ################")
    model_df = {m: df for (m, d), df in group_cal_store.items() if d == dim_name}
    plot_crc_metric_boxplots(model_df,
                             title_prefix=f"Group-level CRC (by {dim_name})")

Applying **per-group** (Mondrian) thresholds but measuring the metrics on the
whole pool, the results are **almost identical to RQ1**: the global FNR is still
$\approx \alpha$ (0.045–0.047, 0.093–0.098, 0.143–0.147, 0.193–0.195,
0.244–0.245), with recall and precision in the same range. The only visible
difference is **at high `alpha`**, where partitioning the calibration costs a
fraction of a point: at `alpha=0.25` accuracy decreases slightly (sector ~0.69,
cohort ~0.70) relative to the single marginal threshold of RQ1 (~0.71), and the
FPR rises correspondingly (sector ~0.33, cohort ~0.32 vs ~0.31 marginal). The
**cohort** calibration stays closer to RQ1 than the **sector** one. The
per-metric boxplots confirm seed distributions that essentially overlap with the
global ones.

Conditional control is therefore **almost free**: it costs at most a fraction of
a point of aggregate accuracy/FPR at high `alpha`, with no substantial
"fairness vs. performance" trade-off.


### Direct comparison — metric deltas (Mondrian − standard CRC)

Summary table of the **differences** between the **standard CRC** version (RQ1,
single global threshold) and the **Mondrian** version (RQ3, per-subgroup
threshold), for each model and each `alpha`, separately for the two partitions
(sector and cohort). Values are means over seeds; $\Delta = \text{Mondrian} -
\text{standard}$.

Sign convention: for **FNR/FPR** a positive $\Delta$ means Mondrian incurs *more*
error; for **accuracy/precision/recall/f1** a positive $\Delta$ means Mondrian is
*better*. Expected pattern: $\Delta\,\text{FNR} \approx 0$ (both pin the FNR to
the same `alpha`) and $\Delta\,\text{FPR} > 0$ / $\Delta\,\text{accuracy} < 0$ at
high `alpha` (the aggregate, fraction-of-a-point cost of equalizing across
groups). A $\Delta$ that is **not uniformly zero** also verifies that the two
versions do not use the same thresholds.


In [ ]:
# Δ metriche: Mondrian (group-level, RQ3) − CRC standard (marginal, RQ1).
# Standard  -> conformal_store[model]                 (single global threshold)
# Mondrian  -> group_cal_store[(model, grouping)]     (per-group thresholds)
# Same fixed pool and same seeds, so the two are directly comparable row by row.
metric_cols = ["accuracy", "precision", "recall", "f1", "FNR", "FPR"]

def _conf_means(df):
    """Mean-over-seeds conformal metrics indexed by target alpha."""
    return df[df["calibrated"]].groupby("alpha")[metric_cols].mean()

crc_mondrian_delta = {}
for dim_name, _ in subgroup_dims:
    blocks = []
    for model_type in conformal_store:
        std  = _conf_means(conformal_store[model_type])
        mond = _conf_means(group_cal_store[(model_type, dim_name)])
        delta = (mond - std)
        delta.insert(0, "model", model_type.upper())
        blocks.append(delta.reset_index())
    tab = (pd.concat(blocks, ignore_index=True)[["model", "alpha"] + metric_cols]
             .set_index(["model", "alpha"])
             .round(4))
    crc_mondrian_delta[dim_name] = tab
    print(f"################  Δ (Mondrian − standard CRC) — grouping: {dim_name}  ################")
    display(tab)

### As RQ2 — subgroups under group-level calibration

The per-subgroup tables and plots, now with the **conditional** (Mondrian)
threshold: `FNR_cond` is pulled back toward `alpha`, with `FNR_marg` kept as
reference. `infeasible` flags groups with too few positive calibration examples.
Industry sector first, then founding-year cohort (with the per-model line plot).

In [ ]:
# Industry sector — conditional table then grouped boxplot.
cond_cols = ["group", "alpha", "FNR_marg", "FNR_cond", "FNR_cond - alpha",
             "n_pool", "n_pos_pool", "infeasible"]
dim_name = "Industry sector"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} (group-calibrated) ===")
    display(tab[cond_cols].rename(columns={"group": dim_name}))

sector_cond = {m: raw for (m, d), raw in subgroup_raw_cond.items() if d == dim_name}
plot_fnr_by_group_boxplots(sector_cond, dim_name, model_color,
                           ylabel="realized FNR (group-calibrated)")

With a **sector-specific** threshold, FNR_cond is re-aligned toward `alpha` and
the RQ2 gap closes almost everywhere (an empty `FNR_cond − alpha` cell = group
at/below target). In the boxplot the sector boxes, previously scattered above and
below the red line, **concentrate around the target**:

- **B2C**, previously the most under-covered, is now controlled (FNR_cond
  ~0.047→0.242, gap closed): its threshold is **lowered** as needed, so positives
  are predicted more often and false negatives fall to `alpha`.

- The previously **over-covered** sectors (Financial Services 0.008→~0.033,
  Energy, Healthcare) see their FNR **rise toward `alpha`**: their threshold is
  **raised** (positives predicted less often), yielding fewer false positives and
  higher precision, at the cost of giving up excess conservatism. This is a
  **transfer of protection** from easy groups to hard ones.

- Small residual exceedances remain (B2B +0.003–0.008; Materials +0.008–0.016),
  consistent with finite-sample noise in the less populated groups.

This is **group-conditional / equalized** coverage in action: risk is made
homogeneous across sectors instead of being guaranteed only on average. Explicit
trade-off: risk equity across groups $\rightleftarrows$ a slight loss of
conservatism where it was previously excessive.


In [ ]:
# YearFounded cohort — conditional table, grouped boxplot, per-model line plot.
cond_cols = ["group", "alpha", "FNR_marg", "FNR_cond", "FNR_cond - alpha",
             "n_pool", "n_pos_pool", "infeasible"]
dim_name = "YearFounded cohort"
for (model_type, d), tab in subgroup_tables.items():
    if d != dim_name:
        continue
    print(f"=== {model_type.upper()} — {dim_name} (group-calibrated) ===")
    display(tab[cond_cols].rename(columns={"group": dim_name}))

cohort_cond = {m: raw for (m, d), raw in subgroup_raw_cond.items() if d == dim_name}
plot_fnr_by_group_boxplots(cohort_cond, dim_name, model_color,
                           ylabel="realized FNR (group-calibrated)")
plot_fnr_by_group_lines(cohort_cond, dim_name, model_color,
                        ylabel="realized FNR (group-calibrated)")

The per-cohort threshold aligns FNR_cond to `alpha` across all years and, in
particular, **closes the gap of the 2017 cohort** seen in RQ2 (FNR_cond
~0.048→0.253, gap now closed at almost every `alpha`). The line plot becomes much
"flatter" around the target line than in RQ2, and in the boxplot the cohorts
tighten onto the red line.

**The limit: infeasible groups.** The **2008 cohort is flagged
`infeasible=True`**: with only **26 positives in the pool** (~13 in the
calibration half) there are not enough positive examples to calibrate a reliable
per-group threshold at the requested `alpha` level. For that group the
conditional coverage must be considered **invalid**, regardless of the realized
value.

**Interpretation.** The message is twofold:

- group-level (Mondrian) CRC **removes the temporal disparity** where groups are
  sufficiently populated;

- on tiny groups the conditional guarantee is **not attainable** and must be
  handled explicitly (fallback to the marginal threshold, merging cohorts, or a
  more permissive target).


## Summary

- **RQ1 — Marginal control works.** By choosing a risk level `alpha`, Conformal
  Risk Control brings the test FNR to $\mathbb{E}[\mathrm{FNR}] \approx \alpha$
  (~0.048, 0.097, 0.147, 0.194, 0.245) for all three models, with low variance
  across seeds. The guarantee is a property of the **procedure**, not of the
  classifier (RF, LGB, MLP almost indistinguishable), and comes with an explicit,
  *tunable* trade-off: more protection against FNs $\Rightarrow$ more FPs, lower
  precision/accuracy. `alpha` is the knob.

- **RQ2 — "Marginal" does not mean "for everyone".** The same global threshold
  leaves some subgroups uncovered: **B2C** (and partly B2B) among sectors, and
  the **recent 2017 cohort** among founding years, remain above `alpha`; other
  groups are instead over-protected. This is the known gap between **marginal**
  and **conditional** coverage: an average satisfied on the pool hides systematic
  disparities.

- **RQ3 — Group-level (Mondrian) calibration closes the gap, almost for free.** A
  per-group threshold re-aligns the conditional FNR toward `alpha` (the B2C and
  2017 gaps are reabsorbed), *without* worsening the aggregate metrics beyond a
  fraction of a point at high `alpha`. It is a **transfer of protection** from
  easy groups to hard ones. The limit is finite-sample: groups with too few
  positives (e.g. the **2008** cohort, 26 positives) are `infeasible`, and for
  them the conditional guarantee is not attainable.
